# LangChain Multi-proveedor (Moderno)

## Objetivos de Aprendizaje
- Implementar una fábrica que seleccione dinámicamente el proveedor (OpenRouter o Google AI Studio).
- Utilizar el enfoque moderno de LangChain para memoria (`RunnableWithMessageHistory`) y cadenas (`RunnableSequence`).
- Demostrar la abstracción de LangChain: el código de negocio no cambia al cambiar de proveedor.
- Probar streaming, memoria y cadenas con diferentes backends.

## Configuración

Define la variable de entorno `PROVIDER` con valor `openrouter` o `google` para elegir el backend. Si no se define, se usará `openrouter` por defecto.

In [1]:
!pip install langchain langchain-openai langchain-core python-dotenv -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 2.5 MB/s eta 0:00:00


In [2]:
import os
from dotenv import load_dotenv
from getpass import getpass

load_dotenv()

# Determinar proveedor
PROVIDER = os.environ.get("PROVIDER", "openrouter").lower()
if PROVIDER not in ["openrouter", "google"]:
    raise ValueError("PROVIDER debe ser 'openrouter' o 'google'")

print(f"🔧 Proveedor seleccionado: {PROVIDER.upper()}")

# Configurar credenciales según proveedor (solo establece variables de entorno)
if PROVIDER == "openrouter":
    if not os.environ.get("OPENROUTER_API_KEY"):
        clave = getpass("Ingresa tu clave de OpenRouter: ")
        os.environ["OPENROUTER_API_KEY"] = clave
    BASE_URL = "https://openrouter.ai/api/v1"
    MODELO_DEFECTO = "minimax/minimax-m3:free"  # gratuito y disponible
else:  # google
    if not os.environ.get("GEMINI_API_KEY"):
        clave = getpass("Ingresa tu clave de Google AI Studio: ")
        os.environ["GEMINI_API_KEY"] = clave
    BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
    MODELO_DEFECTO = "gemini-3.6-flash"  # modelo actualizado (ya no disponible 2.5)

print(f"✅ Clave configurada para {PROVIDER.upper()}")
print(f"Modelo por defecto: {MODELO_DEFECTO}")

🔧 Proveedor seleccionado: OPENROUTER
Ingresa tu clave de OpenRouter: ··········
✅ Clave configurada para OPENROUTER
Modelo por defecto: minimax/minimax-m3:free


In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import warnings
warnings.filterwarnings('ignore')

def crear_llm(modelo=MODELO_DEFECTO, temperature=0.7, max_tokens=300):
    # Obtener clave directamente del entorno
    if PROVIDER == "openrouter":
        api_key = os.environ.get("OPENROUTER_API_KEY")
    else:
        api_key = os.environ.get("GEMINI_API_KEY")

    if not api_key:
        raise ValueError(f"❌ Clave para {PROVIDER.upper()} no encontrada en el entorno.")

    kwargs = {
        "base_url": BASE_URL,
        "api_key": api_key,
        "model": modelo,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    if PROVIDER == "openrouter":
        kwargs["default_headers"] = {
            "HTTP-Referer": "http://localhost:8000",
            "X-Title": "LangChain Multi-proveedor",
        }
    return ChatOpenAI(**kwargs)

llm = crear_llm()

# Prueba de conexión
response = llm.invoke([HumanMessage(content="Hola, responde en una palabra.")])
print(f"✅ Conexión exitosa con {PROVIDER.upper()}")
print(f"Respuesta: {response.content}")
if hasattr(response, 'usage_metadata'):
    print(f"Tokens: {response.usage_metadata.get('total_tokens', 'N/A')}")

✅ Conexión exitosa con OPENROUTER
Respuesta: Hola.
Tokens: 173


## Ejemplos Unificados

Los siguientes ejemplos funcionan con cualquier proveedor gracias a la abstracción de LangChain.

In [4]:
# 1. Streaming
print("=== Streaming ===")
print("🔹 ", end="")
for chunk in llm.stream([HumanMessage(content="Escribe un refrán corto.")]):
    print(chunk.content, end="", flush=True)
print("\n" + "-"*40)

=== Streaming ===
🔹 Aquí tienes uno:

**"Más vale prevenir que curar."**

Es breve, conocido y aplicable a muchas situaciones de la vida cotidiana. ¿Quieres uno sobre algún tema en particular?
----------------------------------------


In [5]:
# 2. Memoria moderna con RunnableWithMessageHistory
prompt_template = ChatPromptTemplate.from_messages([
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])
chain = prompt_template | llm | StrOutputParser()

store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

def demo_memoria():
    session_id = "sesion_multi"
    print("=== Memoria Conversacional (Moderno) ===")

    print("\n[Usuario] Mi nombre es Laura y soy bióloga.")
    r1 = chain_with_memory.invoke(
        {"input": "Mi nombre es Laura y soy bióloga."},
        config={"configurable": {"session_id": session_id}}
    )
    print(f"[Asistente] {r1}")

    print("\n[Usuario] ¿Cuál es mi profesión?")
    r2 = chain_with_memory.invoke(
        {"input": "¿Cuál es mi profesión?"},
        config={"configurable": {"session_id": session_id}}
    )
    print(f"[Asistente] {r2}")

demo_memoria()

=== Memoria Conversacional (Moderno) ===

[Usuario] Mi nombre es Laura y soy bióloga.
[Asistente] ¡Hola, Laura! Encantado de conocerte. Es fascinante que te dediques a la biología, un campo tan diverso y fundamental para entender la vida en nuestro planeta.

¿En qué rama de la biología te especializas? Por ejemplo, ¿trabajas en ecología, genética, microbiología, biología marina, conservación, o alguna otra área?

Si hay algo en lo que pueda ayudarte —ya sea una consulta científica, información sobre algún tema biológico, o cualquier otra cosa— no dudes en preguntar.

[Usuario] ¿Cuál es mi profesión?
[Asistente] Según lo que me compartiste en tu primer mensaje, tu profesión es **bióloga**.

¿En qué puedo ayudarte hoy?


In [6]:
# 3. Cadena con RunnableSequence
def demo_cadena():
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Eres un asistente útil y conciso."),
        ("human", "{pregunta}")
    ])
    chain = prompt | llm | StrOutputParser()

    pregunta = "¿Cuáles son los tres estados del agua?"
    print("=== Cadena Simple ===")
    respuesta = chain.invoke({"pregunta": pregunta})
    print(respuesta)

demo_cadena()

=== Cadena Simple ===
Los tres estados del agua son:

1. **Sólido**: el agua en forma de hielo, con temperaturas por debajo de 0 °C. Sus moléculas se organizan en una estructura cristalina y rígida.

2. **Líquido**: el agua en su forma más común, con temperaturas entre 0 °C y 100 °C (a nivel del mar). Sus moléculas se mueven con libertad, pero mantienen cierto grado de cohesión.

3. **Gaseoso**: el agua en forma de vapor, con temperaturas por encima de 100 °C (a nivel del mar). Sus moléculas se mueven rápida y libremente, llenando todo el espacio disponible.

Los cambios entre estos estados se llaman:
- **Fusión**: sólido → líquido
- **Vaporización/ebullición**: líquido → gas
- **Condensación**: gas → líquido
- **Solidificación/congelación**: líquido → sólido
- **Sublimación**: sólido → gas
- **Deposición**: gas → sólido

¿Te gustaría profundizar en alguno de ellos?


## Comparación de Modelos Multi-proveedor

Dependiendo del proveedor, los modelos disponibles son diferentes. Usamos modelos gratuitos para las pruebas.

**Nota:** Si estás en OpenRouter, usa modelos con `:free`. Si estás en Google, todos los modelos son gratuitos.

In [7]:
# Definir modelos según proveedor (solo gratuitos)
if PROVIDER == "openrouter":
    modelos = {
        "MiniMax M3": "minimax/minimax-m3:free",
        "NVIDIA Nemotron": "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free",
        "Cohere North": "cohere/north-mini-code:free",
        # "Llama 3.1" (si está disponible): "meta-llama/llama-3.1-8b-instruct:free",
    }
else:  # google
    modelos = {
        "Gemini 3.6 Flash": "gemini-3.6-flash",
        "Gemini 2.0 Flash": "gemini-2.0-flash",
        "Gemini 1.5 Pro": "gemini-1.5-pro",
    }

def probar_modelo(model_id, prompt="Explica qué es un algoritmo en una frase."):
    try:
        llm_temp = crear_llm(modelo=model_id, max_tokens=200)
        respuesta = llm_temp.invoke([HumanMessage(content=prompt)])
        print(f"🔹 Modelo: {model_id}")
        print(f"   Respuesta: {respuesta.content}")
        if hasattr(respuesta, 'usage_metadata'):
            print(f"   Tokens: {respuesta.usage_metadata.get('total_tokens', 'N/A')}")
        print("-" * 40)
    except Exception as e:
        print(f"❌ Error con {model_id}: {e}")

print(f"=== Prueba de Modelos en {PROVIDER.upper()} ===\n")
if PROVIDER == "openrouter":
    print("✅ Todos los modelos seleccionados son gratuitos.\n")
else:
    print("✅ Todos los modelos de Google AI Studio son gratuitos.\n")

for nombre, modelo_id in modelos.items():
    probar_modelo(modelo_id)

=== Prueba de Modelos en OPENROUTER ===

✅ Todos los modelos seleccionados son gratuitos.

🔹 Modelo: minimax/minimax-m3:free
   Respuesta: Un algoritmo es un conjunto finito y ordenado de instrucciones o pasos bien definidos que, al seguirse, permiten resolver un problema o realizar una tarea específica.
   Tokens: 205
----------------------------------------
🔹 Modelo: nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free
   Respuesta: Un algoritmo es una secuencia finita y ordenada de pasos que permite resolver un problema o realizar una tarea.
   Tokens: 126
----------------------------------------
🔹 Modelo: cohere/north-mini-code:free
   Respuesta: Un algoritmo es un conjunto finito y ordenado de instrucciones claras que toma una entrada, realiza un procesamiento paso a paso y produce una salida, resolviendo un problema específico o realizando una tarea.
   Tokens: 86
----------------------------------------


## Conclusión

LangChain permite cambiar de proveedor sin modificar la lógica de negocio. Esta fábrica es la base para aplicaciones escalables que puedan adaptarse a diferentes backends según disponibilidad, costo o preferencia del usuario.

Hemos visto cómo utilizar:
- **Streaming** para respuestas en tiempo real.
- **Memoria moderna** con `RunnableWithMessageHistory`.
- **Cadenas** con `RunnableSequence`.
- **Comparación de modelos** entre proveedores.

Este notebook está actualizado para LangChain >=0.3.0 y libre de advertencias de deprecación.